# Synthetic Data Generation: Clinical Notes (Free-Text Clinical Documentation)

**Objective:** Generate clinical visit notes for each existing patient record by emulating the imperfections of real-world EHR text. 

---
---
---

## 1. Setup

---

In [1]:
# Data Management
import json
import numpy as np
import pandas as pd

# Utils
import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from IPython.display import display, HTML, IFrame, Image
from pathlib import Path
import random
from random import choice, choices, randint, randrange, sample, shuffle
pd.set_option('display.max_colwidth', None)

In [2]:
# Define project root and file paths relative to user OS
BASE_DIR = Path.cwd()

SYNTHETIC_PATIENT_DATA_PATH = BASE_DIR / 'simulated_data' / 'uc_new_patients.csv'

TRIAGE_COMPLAINTS_PATH = BASE_DIR / 'simulated_data' / 'text_data' / 'triage_complaints.json'
DIAGNOSTICS_OPTIONS_PATH = BASE_DIR / 'simulated_data' / 'text_data' / 'physician_dx.json'
TREATMENT_OPTIONS_PATH = BASE_DIR / 'simulated_data' / 'text_data' / 'tx_poa.json'
SUPPORTING_TEXT_PATH = BASE_DIR / 'simulated_data' / 'text_data' / 'supporting_text.json'

In [3]:
# Load the possibilities of text options to source from for constructing clinical notes
with TRIAGE_COMPLAINTS_PATH.open('r', encoding='utf-8') as a:
    TRIAGE_COMPLAINTS = json.load(a)
with DIAGNOSTICS_OPTIONS_PATH.open('r', encoding='utf-8') as b:
    DIAGNOSTICS_OPTIONS = json.load(b)
with TREATMENT_OPTIONS_PATH.open('r', encoding='utf-8') as c:
    TREATMENT_OPTIONS = json.load(c)
with SUPPORTING_TEXT_PATH.open('r', encoding='utf-8') as d:
    SUPPORTING_TEXT = json.load(d)

# Load the previously assembled patient visit records
pt_visit_records = pd.read_csv(SYNTHETIC_PATIENT_DATA_PATH, low_memory=False)
pt_visit_records.head()

,pt_id,pt_name,pt_dob,pt_age,visit_location,visit_reason,visit_code,visit_date,visit_day,checkin_time,checkout_time,rolling_ct,rolling_code,assigned_num_techs,needed_num_techs
0,1026728,Alexander Romero,1983-11-21,38,denver,cold/flu/fever,4,2022-05-01,Sunday,08:52:10,09:56:10,1,0.0,3,1
1,1026730,Kayla Wells,1967-05-15,55,denver,vaccination,3,2022-05-01,Sunday,09:01:47,09:27:47,2,4.0,3,1
2,1026729,Michael Hernandez,1957-06-16,64,denver,cuts/abscess,5,2022-05-01,Sunday,09:09:46,09:59:46,3,3.5,3,1
3,1026747,Donald Sawyer,1993-05-08,29,denver,UTI,4,2022-05-01,Sunday,09:27:52,10:13:52,3,4.5,3,1
4,1026738,Andrea Adams,1989-02-22,33,denver,stomach-pain,5,2022-05-01,Sunday,10:06:42,11:30:42,1,0.0,3,1


---
---
## 2: Documentation Generation: Triage & Physician Notes

**Objective:** Generate synthetic triage notes and physician documentation for each of the exisiting patient visit records with supporting JSON documents.


---

#### 2a: Explore the pre-constructed `visit_reason` data to help brainstorm the clinical note generation process.

In [4]:
# Examine the number of patient complaints in the dataset
pt_visit_records.visit_reason.value_counts()

cold/flu/fever        1384
covid-test             889
vaccination            699
cough                  687
ear-pain               635
chest-pain             582
ache/pain              581
sore-throat            579
stomach-pain           499
UTI                    445
lab-work               417
drug-test              372
weakness/dizziness     322
rash/allergy           291
headache               270
cuts/abscess           172
injury/accident        167
physical               104
diarrhea                97
pink-eye                82
Name: visit_reason, dtype: int64

For each of these reasons, it would make sense to come up with a `dict` of related complaints to give a set of possible combinations to output as part of the triage notes. 
Similarly, a set of treatment plans and diagnositic measures for each one would serve as the sampling source for the data generation.
With that, ancillary info such as the already-existing patient age/gender can be incorporated as part of the notes. 

#### 2b: Set up a source of possible note options for each of the patient complaints.

Since the lists of dictionaries will be too long to hard-code in this notebook, they were constructured and saved as JSON files and imported in Section 1. Next, we can explore that data and brainstorm ways to put the pieces together as part of the three text features.

#### 2c: Compile the text information to generate appropriate triage notes.

In [ ]:
# Isolate the exceptions for supporting text generation
NON_SYMPTOM_VISITS = [
    'lab-work',
    'drug-test',
    'vaccination',
    'covid-test'
]

def generate_triage_note(visit_reason, age):
    """
    Generate synthetic triage note from visit reason and patient age.
    """

    # Pull complaint options for the specific visit reason
    complaint_options = TRIAGE_COMPLAINTS[visit_reason]

    # Sample chief complaint
    primary_complaint = choice(
        complaint_options['primary']
    )

    # Sample associated symptoms
    if visit_reason in ['lab-work', 'drug-test', 'vaccination']:
        secondary_complaints = []

    else:
        secondary_count = randint(0, 3)

        if secondary_count > 0:
            secondary_complaints = sample(
                complaint_options['secondary'],
                k=min(secondary_count, len(complaint_options['secondary']))
            )
        else:
            secondary_complaints = []

    # Pull supporting text options
    triage_text = SUPPORTING_TEXT['triage']

    # Generate opening phrase
    if visit_reason in NON_SYMPTOM_VISITS:
        opening = 'here for'
    else:
        opening = choice(
            triage_text['opening']
        )

    # Generate severity descriptor
    severity = choice(
        triage_text['severity_descriptors']
    )

    # Generate duration
    duration_connector = choice(
        triage_text['duration_connector']
    )

    duration = choice(
        triage_text['duration_values']
    )

    # Build note
    age_format = choice([
        f'{age} year old',
        f'{age} yr old'
    ])

    if visit_reason in NON_SYMPTOM_VISITS:
        note = (
            f'{age_format} patient {opening} '
            f'{primary_complaint}'
        )
    else:
        note = (
            f'{age_format} patient {opening} '
            f'{severity} {primary_complaint} '
            f'{duration_connector} {duration}'
        )

    # Add secondary complaints if present
    if secondary_complaints:
        transition = choice(
            triage_text['transition_words']
        )

        note += f' {transition} ' + ', '.join(secondary_complaints)

    # Add negative symptoms occasionally
    if visit_reason not in ['lab-work', 'drug-test', 'vaccination'] and random.random() < 0.5:
        negative_connector = choice(
            triage_text['negative_connectors']
        )

        negative_symptoms = sample(
            triage_text['common_negatives'],
            k=randint(1, 2)
        )

        note += f'. Patient {negative_connector} ' + ', '.join(negative_symptoms)

    note += '.'

    return note

In [6]:
# Generate triage notes for all patient records
pt_visit_records['triage_notes'] = pt_visit_records.apply(
    lambda row: generate_triage_note(
        row['visit_reason'],
        row['pt_age']
    ),
    axis=1
)

In [7]:
pt_visit_records.triage_notes[0:10]

0                                                                           38 year old patient presents with moderate fever over the past two days also experiencing loss of appetite, runny nose.
1                                                                                                                                                            55 year old patient here for flu shot.
2                                                                                          64 yr old patient arrives with severe skin abscess from about one day associated with redness, bleeding.
3                                            29 yr old patient presents with worsening possible urinary tract infection since one day also experiencing lower abdominal discomfort, blood in urine.
4                                  33 year old patient complains of persistent abdominal discomfort from about three days also experiencing vomiting. Patient denies vomiting, shortness of breath.
5                   

#### 2d: Compile the text information to generate appropriate diagnostic plan notes.

In [8]:
def generate_diagnostic_note(visit_reason):
    """
    Generate synthetic physician diagnostic note from visit reason.
    """

    # Pull diagnostic options for the specific visit reason
    diagnostic_options = DIAGNOSTICS_OPTIONS[visit_reason]

    # Pull supporting text options
    diagnostic_text = SUPPORTING_TEXT['diagnostics']

    # Generate diagnostic action phrase
    diagnostic_action = choice(
        diagnostic_text['actions']
    )

    # Randomly decide if no diagnostic testing was performed
    if diagnostic_options['no_testing'] and random.random() < 0.15:
        note = choice(
            diagnostic_options['no_testing']
        )

    else:
        # Sample diagnostic test/evaluation
        diagnostic_test = choice(
            diagnostic_options['tests']
        )

        # Determine result type
        if 'screening' in diagnostic_test.lower() or 'assessment' in diagnostic_test.lower():
            diagnostic_result = choice(
                diagnostic_text['screening_results']
            )

        else:
            diagnostic_result = choice(
                diagnostic_text['positive_results']
            )

        # Build diagnostic note
        note = (
            f'{diagnostic_action} '
            f'{diagnostic_test}, which {diagnostic_result}'
        )

    return note + '.'

In [9]:
# Generate diagnostic notes for all patient records
pt_visit_records['diagnostic_notes'] = (
    pt_visit_records['visit_reason']
    .apply(generate_diagnostic_note)
)

In [10]:
pt_visit_records['diagnostic_notes'][0:10]

0                                         Testing included rapid strep test, which was negative.
1                        Workup included vital signs assessment, which was within normal limits.
2                                            Testing included X-ray imaging, which was negative.
3                     Diagnostic evaluation included urinalysis, which showed no acute findings.
4                             Diagnostic evaluation included pregnancy test, which was negative.
5                        Workup included vital signs assessment, which was within normal limits.
6    Diagnostic evaluation included basic vital sign assessment, which was within normal limits.
7                                 Diagnostic evaluation included blood work, which was negative.
8                        Testing included COVID-19 antigen test, which showed no acute findings.
9               Diagnostic evaluation included basic vital sign assessment, which was completed.
Name: diagnostic_notes, dtype:

#### 2e: Compile the text information to generate appropriate treatment plan notes.

In [11]:
def generate_treatment_note(visit_reason):
    """
    Generate synthetic physician treatment note from visit reason.
    """

    # Pull treatment options for the specific visit reason
    treatment_options = TREATMENT_OPTIONS[visit_reason]

    # Pull supporting text options
    treatment_text = SUPPORTING_TEXT['treatment_plan']

    # Generate treatment action phrase
    treatment_action = choice(
        treatment_text['actions']
    )

    # Sample treatment(s)
    treatment_count = randint(1, 2)

    treatments = sample(
        treatment_options['treatments'],
        k=min(
            treatment_count,
            len(treatment_options['treatments'])
        )
    )

    # Build treatment section
    note = (
        f'{treatment_action} '
        f'{", ".join(treatments)}.'
    )

    # Add disposition separately
    if treatment_options['disposition']:

        disposition = choice(
            treatment_options['disposition']
        )

        disposition_action = choice(
            treatment_text['disposition_actions']
        )

        note += (
            f' {disposition_action} '
            f'{disposition}.'
        )

    return note

In [12]:
# Generate treatment notes for all patient records
pt_visit_records['treatment_notes'] = (
    pt_visit_records['visit_reason']
    .apply(generate_treatment_note)
)

In [13]:
pt_visit_records['treatment_notes'][0:5]

0                              Plan included supportive care instructions provided. Patient is ready for discharge with follow-up instructions.
1        Management included post-vaccination monitoring performed. Follow-up plan includes discharge planned with routine follow-up as needed.
2    Management included laceration repaired with sutures, wound cleaned and irrigated. Patient is appropriate for outpatient wound management.
3           Management included hydration encouraged, urinary analgesic prescribed. Patient is ready for discharge with follow-up instructions.
4                                       Management included oral hydration encouraged. Patient is discharge planned with abdominal precautions.
Name: treatment_notes, dtype: object

---
---
## 3: Noise Introduction

**Objective:** Introduce textual imperfections into generated notes that emulate real-world EHR records.


---

In [14]:
# Character-level typo generator
char_aug = nac.KeyboardAug(
    aug_char_p=0.08
)


def add_punctuation_noise(note):
    """
    Add punctuation imperfections.
    """

    noise_type = choice([
        'remove_period',
        'remove_comma',
        'extra_space',
        'double_space'
    ])

    if noise_type == 'remove_period':
        if note.endswith('.'):
            return note[:-1]

    elif noise_type == 'remove_comma':
        if ',' in note:
            return note.replace(',', '', 1)

    elif noise_type == 'extra_space':
        words = note.split()

        if len(words) > 3:
            insert_position = random.randint(1, len(words) - 1)
            words.insert(insert_position, '')

        return ' '.join(words)

    elif noise_type == 'double_space':
        return note.replace(' ', '  ', 1)

    return note


def add_capitalization_noise(note):
    """
    Add occasional single-letter capitalization errors.
    """

    characters = list(note)

    # Only modify alphabetic characters
    letter_positions = [
        i for i, char in enumerate(characters)
        if char.isalpha()
    ]

    if letter_positions:
        # Select one or two letters to modify
        change_count = 1 if random.random() < 0.8 else 2

        selected_positions = sample(
            letter_positions,
            k=min(change_count, len(letter_positions))
        )

        for position in selected_positions:
            characters[position] = characters[position].swapcase()

    return ''.join(characters)


def add_text_noise(note):
    """
    Add realistic EHR-style text imperfections.
    """

    # Ensure input is always treated as text
    if not isinstance(note, str):
        return note

    # Apply noise to 30% of records
    if random.random() > 0.30:
        return note

    # Sometimes apply a second minor imperfection
    noise_count = 2 if random.random() < 0.15 else 1

    for _ in range(noise_count):

        noise_type = choice([
            'typo',
            'punctuation',
            'capitalization'
        ])

        if noise_type == 'typo':
            note = char_aug.augment(note)[0]

        elif noise_type == 'punctuation':
            note = add_punctuation_noise(note)

        elif noise_type == 'capitalization':
            note = add_capitalization_noise(note)

    return note

In [15]:
# Apply the noise into each text feature
pt_visit_records['triage_notes'] = (
    pt_visit_records['triage_notes']
    .apply(add_text_noise)
)

pt_visit_records['diagnostic_notes'] = (
    pt_visit_records['diagnostic_notes']
    .apply(add_text_noise)
)

pt_visit_records['treatment_notes'] = (
    pt_visit_records['treatment_notes']
    .apply(add_text_noise)
)

In [16]:
pt_visit_records['triage_notes'][55:65]

55                                                                                                                      76 yr old pqtient he3e for paboratory testing.
56                                              36 year old patient presents with moderate body pain from about one week associated with pain with movement, swelling.
57                                                           6 yr old patient complains of moderate ear pain for one day also reports decreased hearing, ear pressure.
58    23 year old patient reports worsening muscle pain over the past several days also reports pain with movement, fatigue. Patient negative for vomiting, chest pain
59                                                                                                                             18 yr old patient here for vaccinatiOn.
60                                                                                          52 year old patient arrives with worsening body aches from about two days

---
---
## 4: Export

**Objective**: Export generated data as reusable pickle file for NLP feature engineering.


---

In [17]:
pt_visit_records

,pt_id,pt_name,pt_dob,pt_age,visit_location,visit_reason,visit_code,visit_date,visit_day,checkin_time,checkout_time,rolling_ct,rolling_code,assigned_num_techs,needed_num_techs,triage_notes,diagnostic_notes,treatment_notes
0,1026728,Alexander Romero,1983-11-21,38,denver,cold/flu/fever,4,2022-05-01,Sunday,08:52:10,09:56:10,1,0.0,3,1,"38 year old patient presents with moderate fever over the past two days also experiencing loss of appetite, runny nose.","Testing included rapid strep test, which was negative.",Plan Uncluded su(portive care instructions provided. PatieHt is reSdy for fischarge with foll9w - up instructions.
1,1026730,Kayla Wells,1967-05-15,55,denver,vaccination,3,2022-05-01,Sunday,09:01:47,09:27:47,2,4.0,3,1,55 year old patient here for flu shot.,"Workup included vital signs assessment, which was within normal limits.",Management included post-vaccination monitoring performed. Follow-up plan includes discharge planned with routine follow-up as needed.
2,1026729,Michael Hernandez,1957-06-16,64,denver,cuts/abscess,5,2022-05-01,Sunday,09:09:46,09:59:46,3,3.5,3,1,"64 yr old patient arrives with severe skin abscess from about one day associated with redness, bleeding.","Testing included X-ray imaging, which was negative.","Management included laceration repaired with sutures, wound cleaned and irrigated. Patient is appropriate for outpatient wound management."
3,1026747,Donald Sawyer,1993-05-08,29,denver,UTI,4,2022-05-01,Sunday,09:27:52,10:13:52,3,4.5,3,1,"29 yr old patient presents with worsening possible urinary tract infection since one day also experiencing lower abdominal discomfort, blood in urine.","Diagnostic evaluatiOn included urinalysis, which showed no acute findings.","Management included hydration encouraged, urinary analgesic prescribed. Patient is ready for discharge with follow-up instructions."
4,1026738,Andrea Adams,1989-02-22,33,denver,stomach-pain,5,2022-05-01,Sunday,10:06:42,11:30:42,1,0.0,3,1,"33 year old patient complains of persistent abdominal discomfort from about three days also experiencing vomiting. Patient denies vomiting, shortness of breath.","Diagnostic evaluation included pregnancy test, which was negative.",Manage<ent included oral hydration encouraged. Pa6ient is discharge plannRd with sbdominal precautioMs.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9269,5024859,Vicki Collins,2006-09-09,15,lakewood,headache,5,2022-05-31,Tuesday,17:28:52,18:32:52,3,5.0,4,1,15 yr old patient states moderate head pain over the past one day also experiencing light sensitivity.,No diagnostic testing was performed.,"Treatment included hydration encouraged, anti-inflammatory medication recommended. Patient is discharge planned with return precautions."
9270,5024871,Jennifer Fisher,2009-11-01,12,lakewood,UTI,4,2022-05-31,Tuesday,17:34:42,18:28:42,4,5.0,4,2,12 yr old patient complains of worsening painful urination over the past one week also reports blood in urine. Patient negative for chest pain.,"Workup included bladder scan, which showed no acute findings.",Management included urine culture follow-up planned. Patient is discharge planned with culture follow-up.
9271,5024863,Lisa Walker,1970-05-05,52,lakewood,stomach-pain,5,2022-05-31,Tuesday,17:55:09,19:20:33,4,4.7,4,2,"52 year old patient complains of severe stomach pain since one day also experiencing diarrhea, nausea. Patient without chest pain.","Diagnostic evaluation included urinXlysis, whidh was unremarkaHle.",Treatment included dietary modifications discussed. Patient is discharge planned with abdominal precautions.
9272,5024866,Emily Carson,2013-09-20,8,lakewood,drug-test,3,2022-05-31,Tuesday,18:44:02,19:22:02,2,5.0,4,1,8 yr old patient here for urine drug screen.,"Diagnostic evaluation included toxicology screening, which was reviewed","Treatment included screening completed, results sent for review. Follow-up plan includes cleared for discharge after testing completion."


In [18]:
pt_visit_records.to_pickle(BASE_DIR / 'pickled_objects/' / 'pt_visit_records.pkl')